# Module 04 — What did the foundation model actually learn?| | ||---|---|| **Input** | `pp_nerf_m5_k30.ckpt` (the paper's m6, 175M) + 10 events || **Algorithm** | frozen-backbone forward, residual-stream reconstruction, t-SNE || **Output** | `features.npz`, three figures, two probe curves || **Visualization** | t-SNE colored by ground-truth track |The model was pretrained with **no labels** — its only task was to predict each spacepoint's30 nearest neighbours. This module asks what that bought: do the learned representationsknow about tracks?**Runtime:** ~1 min extraction (GPU) + ~4 min analysis (CPU).

In [ ]:
import os, sys, numpy as np, matplotlib.pyplot as pltsys.path.insert(0, '.')from analyze_features import (load, silhouette, knn_purity, embed2d, scatter, LEARNED_W)NPZ = os.environ.get('FM4NPP_FEATURES', 'features.npz')if not os.path.exists(NPZ):    raise SystemExit('run:  python extract_features.py --n_events 10 --out features.npz')d = load(NPZ)seg, evt, z, stream = d['seg'], d['evt'], d['z'], d['stream']L = z.shape[0]print(f'{len(seg):,} points, {L} layers, dim {z.shape[2]}, {len(np.unique(evt))} events')print(f'tracks per event: {[len(np.unique(seg[evt==e][seg[evt==e]!=0])) for e in np.unique(evt)]}')

## 1. The trap: `return_z=True` does not give you the representationThis is the single most important thing in this module, and it is easy to get wrong.Here is the actual forward pass:```pythonfor layer in self.mamba_layers:    z = layer(x)    feature.append(z)      # <-- this is what return_z=True hands you    x = z + x              # <-- the accumulated stream, never returned````feature[i]` is block *i*'s **contribution**, not the representation after *i* blocks. Alate block contributes a small refinement, so on its own it looks like noise. Probing itanswers a question nobody asked.The representation you want is the residual stream:$$x_i = x_0 + \sum_{j<i} z_j$$`x_0` is the embedder output, which `extract_features.py` saves for exactly this reason.The reconstruction is **exact** — here it is, checked against forward hooks on the reallayer inputs.

In [ ]:
# Verified when this module was written, against forward hooks on a small model:##     layer   max |reconstructed - actual|#         1                   0.000e+00#         2                   0.000e+00#         3                   1.907e-06     <- one float32 ULP#         4                   1.907e-06## analyze_features.load() does the reconstruction:#     stream = x0[None] + concat([zeros_like(z[:1]), z.cumsum(0)[:-1]])print('branch z_1     shape', z[0].shape)print('stream x_1     shape', stream[0].shape, ' == x0 exactly:',      np.allclose(stream[0], d['x0']))print('final          shape', d['final'].shape,      ' == x0 + sum(z):', np.allclose(d['final'], d['x0'] + z.sum(0), atol=1e-3))

## 2. Look at itt-SNE of the final residual stream, one panel per event, colored by ground-truth track.Grey is noise (track id 0). The model never saw any of these labels.

In [ ]:
ev_ids = np.unique(evt)[:10]fig, axes = plt.subplots(2, 5, figsize=(16, 6.6))for ax, e in zip(axes.ravel(), ev_ids):    m = evt == e    ntr = len(np.unique(seg[m][seg[m] != 0]))    scatter(ax, embed2d(d['final'][m]), seg[m], f'event {e}  ({m.sum()} pts, {ntr} tracks)')fig.suptitle('m6 final residual stream — colored by true track, grey = noise', fontsize=12)plt.tight_layout(); plt.show()

Tracks come out as **coherent, well-separated groups**, from a model trained without asingle label. That is the foundation-model claim, made visible.Look at the *shape* of the groups: most tracks form smooth one-dimensional filaments ratherthan round blobs. That is physically right — a track *is* a 1-D trajectory through thedetector — and it is a strong hint about why plain clustering metrics will undersell thesefeatures in a moment.

## 3. Stream versus branch, side by sideTop row: the accumulated stream (the right object). Bottom row: individual branch outputs(the wrong object). Same event, same t-SNE settings.

In [ ]:
big = max(ev_ids, key=lambda e: (evt == e).sum())m = evt == bigshow = [0, 3, 7, 11]fig, axes = plt.subplots(2, len(show) + 1, figsize=(17, 7))scatter(axes[0, 0], embed2d(d['points'][m]), seg[m], 'raw (E,$\\eta$,$\\phi$,r)')for ax, li in zip(axes[0, 1:], show):    scatter(ax, embed2d(stream[li][m]), seg[m], f'stream, entering layer {li+1}')scatter(axes[1, 0], embed2d(d['x0'][m]), seg[m], '$x_0$ (embedder output)')for ax, li in zip(axes[1, 1:], show):    scatter(ax, embed2d(z[li][m]), seg[m], f'branch $z_{{{li+1}}}$ only')fig.suptitle(f'event {big} — top: accumulated stream.  bottom: branch outputs alone',             fontsize=12)plt.tight_layout(); plt.show()

The contrast is the lesson. The stream row shows clean filaments at every depth. The branchrow — especially $z_1$ — is confetti. Both come from the same forward pass; only the choiceof what to plot differs.If you had probed `return_z` output directly, you would have concluded the model gets*worse* with depth. Let's quantify that mistake.

## 4. Two probes- **Silhouette** — are same-track points compact and separated? Standard clustering metric.- **kNN purity** — of a point's 5 nearest neighbours in feature space, how many share its  track? Closer to both the pretraining objective (predict your neighbours) and to what a  tracking head exploits.Both are computed per event and averaged. Noise excluded.

In [ ]:
sil_s = [silhouette(stream[i], seg, evt) for i in range(L)]pur_s = [knn_purity(stream[i], seg, evt) for i in range(L)]sil_z = [silhouette(z[i], seg, evt) for i in range(L)]pur_z = [knn_purity(z[i], seg, evt) for i in range(L)]raw   = (silhouette(d['points'], seg, evt), knn_purity(d['points'], seg, evt))x0    = (silhouette(d['x0'], seg, evt),     knn_purity(d['x0'], seg, evt))final = (silhouette(d['final'], seg, evt),  knn_purity(d['final'], seg, evt))print(f'{"representation":<22}{"silhouette":>12}{"kNN purity":>13}')print('-' * 47)print(f'{"raw (E,eta,phi,r)":<22}{raw[0]:>12.4f}{raw[1]:>13.4f}')print(f'{"x0 (embedder)":<22}{x0[0]:>12.4f}{x0[1]:>13.4f}')for i in range(L):    print(f'{"  stream layer " + str(i+1):<22}{sil_s[i]:>12.4f}{pur_s[i]:>13.4f}')print(f'{"final":<22}{final[0]:>12.4f}{final[1]:>13.4f}')

In [ ]:
xs = np.arange(1, L + 1)fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))for ax, (s, b, base_x0, base_raw, lbl) in zip(axes, [        (pur_s, pur_z, x0[1], raw[1], 'kNN purity (k=5)'),        (sil_s, sil_z, x0[0], raw[0], 'silhouette')]):    ax.plot(xs, s, 'o-', lw=2, color='navy', label='stream $x_i$')    ax.plot(xs, b, 's--', lw=1.6, color='darkorange', label='branch $z_i$')    ax.axhline(base_x0, ls=':', c='green', lw=1.8, label='$x_0$ embedder')    ax.axhline(base_raw, ls='--', c='crimson', lw=1.6, label='raw coords')    ax.set_xlabel('layer'); ax.set_ylabel(lbl); ax.set_xticks(xs)    ax.grid(alpha=.3); ax.legend(fontsize=8)plt.tight_layout(); plt.show()

### What the numbers sayThree things, and none of them is the story you'd expect.**The embedder does most of the work.** Raw coordinates score 0.896 on kNN purity; theembedder output $x_0$ — a NeRF positional encoding, *no Mamba layers at all* — scores 0.956.That single step captures most of the local structure.**Twelve Mamba layers add about +0.010, and peak in the middle.** The stream improves from0.956 to 0.966 by layer 4–5, then declines to 0.959 by layer 12. The final representation isbarely better than the embedder output on this metric.**The branch outputs would have told you the opposite.** $z_{12}$ scores 0.026 onsilhouette — *below the raw coordinates*. Probe the wrong object and you conclude the modeldegrades with depth.

## 5. Does the downstream head know which layers are good?The track-finding adapter doesn't use the last layer. It learns a softmax over all twelve(`weighted_avg_weights`) and takes a weighted average. If our probes measure something thehead cares about, its learned weights should correlate with them.`LEARNED_W` below is the real thing — read off a converged 70,000-event m6 track-finding run.

In [ ]:
from scipy.stats import pearsonr, spearmanrr, p = pearsonr(LEARNED_W, pur_s)rs, ps = spearmanr(LEARNED_W, pur_s)fig, ax = plt.subplots(figsize=(7, 3.8))ax.bar(xs, LEARNED_W, color='slateblue', alpha=.85)ax.axhline(1 / L, ls='--', c='k', lw=1.4, label=f'uniform = {1/L:.4f}')ax.set_xlabel('layer'); ax.set_ylabel('softmax weight'); ax.set_xticks(xs)ax.set_title('Layer weights the m6 track-finding head actually learned')ax.legend(); ax.grid(alpha=.3, axis='y')plt.tight_layout(); plt.show()print(f'pearson  r = {r:+.3f}  (p = {p:.2f})')print(f'spearman   = {rs:+.3f}  (p = {ps:.2f})')print(f'first-6 weight share: {LEARNED_W[:6].sum():.3f}   (uniform = 0.500)')print(f'entropy: {-(LEARNED_W*np.log(LEARNED_W)).sum():.4f}   (uniform = {np.log(L):.4f})')

**No correlation** — r = −0.28 with p = 0.38. The head's weights are close to uniform(entropy 2.472 against a uniform 2.485) with a mild tilt toward early layers, and they donot track either probe.This is a real result, not a failed experiment, and it has a practical consequence that wasverified independently: because the layer mixing is nearly uniform and carries littleinformation, **freezing those 12 weights costs nothing**. Two otherwise identical40-epoch m6 runs:| layer mixing | best ARI₂ ||---|---|| learned (12 weights, free) | 0.8480 || frozen (collapsed to 1 vector) | 0.8508 |Within seed noise. That is what makes the `--combine_layers` cache in the official repopossible: collapsing 12 layers to 1 at cache time shrinks storage **12×** — from 2.17 TB to185 GB for the full 70k training split — with no measurable accuracy cost.

## 6. The caveat that mattersYou could read section 4 as "the Mamba layers barely help." **Don't.** Both probes measure*Euclidean geometry in the raw feature space* — do same-track points end up near each other.The actual track-finding head is a **query decoder with attention and Hungarian matching**.It does not need tracks to be round compact blobs; it needs structure that attention canread out. Those are different requirements, and the filament shapes in section 2 are exactlythe case where a compactness metric undersells a representation.The evidence that scale genuinely helps, from held-out track reconstruction on 6,943 events:| backbone | params | pooled ARI ||---|---|---|| m3 | 5.3M | 0.8176 || m6 | 175M | **0.8591** |A 33× larger backbone buys +0.042 pooled ARI downstream — while our geometric probes seealmost no difference between layer 1 and layer 12 of that same model.**The lesson for your own work:** an unsupervised probe that looks flat is evidence aboutthe probe as much as about the model. Validate representations on the task you care about.

## Summary1. `return_z=True` returns residual **branch** outputs. Reconstruct the stream with   $x_i = x_0 + \sum_{j<i} z_j$ — exact to one float32 ULP.2. Tracks are clearly visible in t-SNE from a model trained with **no labels**, as smooth   1-D filaments.3. The NeRF embedder captures most of the local structure (0.896 → 0.956 kNN purity); the   twelve Mamba layers add ~+0.010, peaking mid-network.4. The head's learned layer weights are near-uniform and uncorrelated with our probes —   which is why freezing them is free, and why the 12× cache reduction works.5. Geometric probes understate what an attention-based head extracts. Trust the downstream   metric.### Where to go next- Re-run with `--model m3` and compare. Does the 5.3M model show the same layer profile?- Try `k=10, 20` in `knn_purity`. Does the ranking hold at larger neighbourhoods?- Colour by `pid_target` instead of `seg_target` — is particle *type* linearly visible?- The downstream modules (track finding, PID, noise tagging) build directly on these  features.